# Preprocesamiento Avanzado: Forest Cover Type Dataset

## 🎯 Objetivos de Aprendizaje
- **Técnica Principal**: Feature Selection y Manejo de Alta Dimensionalidad
- **Dataset**: Forest Cover Type (54 características)
- **Progresión**: Segundo paso después de Wine Quality

## 📊 Contexto del Dataset
El Forest Cover Type Dataset contiene información sobre diferentes tipos de cobertura forestal en áreas protegidas de Estados Unidos. Con **54 características**, es ideal para aprender:
- Análisis de correlaciones a gran escala
- Feature Selection automático
- Técnicas de reducción de dimensionalidad
- Manejo de la maldición de la dimensionalidad


In [ ]:
# 📦 Importaciones
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectKBest, f_classif, RFE
from sklearn.decomposition import PCA
from sklearn.metrics import classification_report, accuracy_score
from scipy.stats import pearsonr
import warnings
warnings.filterwarnings('ignore')

# Configuración de visualización
plt.style.use('seaborn-v0_8')
sns.set_palette("husl")
plt.rcParams['figure.figsize'] = (12, 8)

## 🔄 Paso 1: Carga y Exploración Inicial

**¿Por qué empezar con exploración?** Con 54 características, es crucial entender la estructura antes de aplicar técnicas de feature selection.

In [ ]:
# Cargar el dataset
print("🌲 Cargando Forest Cover Type Dataset...")
forest_cover = fetch_openml(name='forestcovertype', version=1, as_frame=True)

# Información básica
X = forest_cover.data
y = forest_cover.target

print(f"📊 Shape del dataset: {X.shape}")
print(f"🎯 Número de clases: {len(np.unique(y))}")
print(f"📈 Distribución de clases:")
print(y.value_counts().sort_index())

# Verificar valores faltantes
missing_values = X.isnull().sum()
print(f"\n🔍 Valores faltantes: {missing_values.sum()} total")
if missing_values.sum() > 0:
    print("Columnas con valores faltantes:")
    print(missing_values[missing_values > 0])
else:
    print("✅ Dataset limpio - sin valores faltantes")

## 📋 Paso 2: Análisis de Tipos de Datos y Estructura

**Técnica Clave**: Identificación automática de tipos de características para aplicar preprocesamiento apropiado.

In [ ]:
# Análisis de tipos de datos
print("🔍 Análisis de tipos de datos:")
print(X.dtypes.value_counts())

# Separar características numéricas y categóricas
numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = X.select_dtypes(include=['object', 'category']).columns.tolist()

print(f"\n📊 Características numéricas: {len(numeric_cols)}")
print(f"📊 Características categóricas: {len(categorical_cols)}")

if categorical_cols:
    print("\n🏷️ Características categóricas encontradas:")
    for col in categorical_cols:
        unique_vals = X[col].nunique()
        print(f"  - {col}: {unique_vals} valores únicos")

# Estadísticas descriptivas básicas
print(f"\n📈 Estadísticas básicas:")
print(X.describe())

## 🔍 Paso 3: Análisis de Correlaciones a Gran Escala

**🎯 Técnica Principal**: Análisis de correlaciones para identificar características redundantes antes de aplicar feature selection automático.

**¿Por qué es importante?** Con 54 características, la matriz de correlación puede revelar patrones ocultos y características altamente correlacionadas que podemos eliminar.

In [ ]:
# Análisis de correlaciones (solo para características numéricas)
if len(numeric_cols) > 0:
    print("🔗 Análisis de correlaciones entre características...")
    
    # Calcular matriz de correlación
    correlation_matrix = X[numeric_cols].corr()
    
    # Encontrar correlaciones altas
    high_corr_pairs = []
    threshold = 0.8  # Umbral para correlación alta
    
    for i in range(len(correlation_matrix.columns)):
        for j in range(i+1, len(correlation_matrix.columns)):
            corr_value = abs(correlation_matrix.iloc[i, j])
            if corr_value > threshold:
                col1 = correlation_matrix.columns[i]
                col2 = correlation_matrix.columns[j]
                high_corr_pairs.append((col1, col2, corr_value))
    
    print(f"\n🔍 Encontradas {len(high_corr_pairs)} pares con correlación > {threshold}:")
    for col1, col2, corr in sorted(high_corr_pairs, key=lambda x: x[2], reverse=True)[:10]:
        print(f"  {col1} ↔ {col2}: {corr:.3f}")
    
    # Visualización de matriz de correlación (muestra)
    plt.figure(figsize=(15, 12))
    mask = np.triu(np.ones_like(correlation_matrix, dtype=bool))
    sns.heatmap(correlation_matrix, mask=mask, annot=False, cmap='coolwarm', 
                square=True, linewidths=0.5)
    plt.title('Matriz de Correlación - Características Numéricas')
    plt.tight_layout()
    plt.show()
    
    # Identificar características para eliminar basadas en correlación
    to_remove = set()
    for col1, col2, corr in high_corr_pairs:
        # Eliminar la característica con menos correlación con el target (simulado)
        if col1 not in to_remove and col2 not in to_remove:
            to_remove.add(col2)  # Eliminamos la segunda por convención
    
    print(f"\n🗑️ Características identificadas para eliminación por correlación alta: {len(to_remove)}")
    for col in sorted(to_remove):
        print(f"  - {col}")
else:
    print("⚠️ No se encontraron características numéricas para análisis de correlación")
    to_remove = set()

## 🎯 Paso 4: Feature Selection Automático

**🎯 Técnica Principal**: Aplicación de múltiples técnicas de feature selection para comparar resultados.

**¿Por qué múltiples técnicas?** En datasets de alta dimensionalidad, diferentes métodos pueden seleccionar características diferentes. Es importante entender las fortalezas de cada método.

In [ ]:
# Preparar datos para feature selection
X_processed = X.copy()
y_encoded = LabelEncoder().fit_transform(y) # Codificar etiquetas

# Dividir datos
X_train, X_test, y_train, y_test = train_test_split(
    X_processed, y_encoded, test_size=0.2, random_state=42, stratify=y_encoded
)

# Escalar características
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"📊 Datos preparados para feature selection:")
print(f"  - Características originales: {X_processed.shape[1]}")
print(f"  - Tamaño entrenamiento: {X_train_scaled.shape}")
print(f"  - Tamaño prueba: {X_test_scaled.shape}")

# Función para evaluar performance
def evaluate_features(X_train, X_test, y_train, y_test, feature_names):
    """Evalúa el rendimiento con las características seleccionadas"""
    rf = RandomForestClassifier(n_estimators=100, random_state=42)
    rf.fit(X_train, y_train)
    
    train_score = rf.score(X_train, y_train)
    test_score = rf.score(X_test, y_test)
    
    return train_score, test_score

# Evaluación baseline (todas las características)
baseline_train, baseline_test = evaluate_features(
    X_train_scaled, X_test_scaled, y_train, y_test, X_processed.columns
)
print(f"\n📈 Performance baseline (todas las características):")
print(f"  - Entrenamiento: {baseline_train:.4f}")
print(f"  - Prueba: {baseline_test:.4f}")

In [ ]:
# Técnica 1: SelectKBest con f_classif
print("\n🎯 TÉCNICA 1: SelectKBest (univariado)")
k_features = [5, 10, 15, 20, 25, 30]  # Diferentes números de características
results_selectk = []

for k in k_features:
    selector = SelectKBest(score_func=f_classif, k=k) # Selección de k mejores
    X_train_selected = selector.fit_transform(X_train_scaled, y_train)
    X_test_selected = selector.transform(X_test_scaled)
    
    train_score, test_score = evaluate_features(
        X_train_selected, X_test_selected, y_train, y_test, 
        X_processed.columns[selector.get_support()]
    )
    
    results_selectk.append({
        'k': k,
        'train_score': train_score,
        'test_score': test_score,
        'selected_features': X_processed.columns[selector.get_support()].tolist()
    })
    
    print(f"  k={k}: Test Accuracy = {test_score:.4f}")

# Mejor resultado SelectKBest
best_selectk = max(results_selectk, key=lambda x: x['test_score'])
print(f"\n🏆 Mejor SelectKBest: k={best_selectk['k']} con accuracy {best_selectk['test_score']:.4f}")
print(f"Características seleccionadas: {best_selectk['selected_features'][:10]}...")  # Mostrar solo las primeras 10

In [ ]:
# Técnica 2: Recursive Feature Elimination (RFE)
print("\n🎯 TÉCNICA 2: Recursive Feature Elimination (RFE)")
print("⚠️ Nota: RFE puede ser lento con muchas características")

# Probar con diferentes números de características
n_features_to_test = [10, 20, 30]
results_rfe = []

for n_features in n_features_to_test:
    try:
        print(f"  Probando con {n_features} características...")
        
        # Usar RandomForest como estimador base
        rf_estimator = RandomForestClassifier(n_estimators=50, random_state=42)
        rfe = RFE(estimator=rf_estimator, n_features_to_select=n_features)
        
        X_train_rfe = rfe.fit_transform(X_train_scaled, y_train)
        X_test_rfe = rfe.transform(X_test_scaled)
        
        train_score, test_score = evaluate_features(
            X_train_rfe, X_test_rfe, y_train, y_test,
            X_processed.columns[rfe.support_]
        )
        
        results_rfe.append({
            'n_features': n_features,
            'train_score': train_score,
            'test_score': test_score,
            'selected_features': X_processed.columns[rfe.support_].tolist(),
            'feature_ranking': rfe.ranking_
        })
        
        print(f"    Test Accuracy = {test_score:.4f}")
        
    except Exception as e:
        print(f"    Error con {n_features} características: {str(e)}")
        continue

# Mejor resultado RFE
if results_rfe:
    best_rfe = max(results_rfe, key=lambda x: x['test_score'])
    print(f"\n🏆 Mejor RFE: {best_rfe['n_features']} características con accuracy {best_rfe['test_score']:.4f}")
    print(f"Características seleccionadas: {best_rfe['selected_features'][:10]}...")
else:
    print("⚠️ RFE falló en todas las configuraciones probadas")

In [ ]:
# Técnica 3: Random Forest Feature Importance
print("\n🎯 TÉCNICA 3: Random Forest Feature Importance")

# Entrenar Random Forest y obtener importancias
rf_full = RandomForestClassifier(n_estimators=100, random_state=42)
rf_full.fit(X_train_scaled, y_train)

# Obtener importancias
feature_importance = pd.DataFrame({
    'feature': X_processed.columns,
    'importance': rf_full.feature_importances_
}).sort_values('importance', ascending=False)

print("Top 20 características más importantes:")
print(feature_importance.head(20))

# Probar con diferentes números de características basadas en importancia
importance_thresholds = [0.01, 0.005, 0.003, 0.002]
results_importance = []

for threshold in importance_thresholds:
    selected_features = feature_importance[feature_importance['importance'] >= threshold]['feature'].tolist()
    
    if len(selected_features) > 0:
        feature_indices = [X_processed.columns.get_loc(feature) for feature in selected_features]
        
        X_train_imp = X_train_scaled[:, feature_indices]
        X_test_imp = X_test_scaled[:, feature_indices]
        
        train_score, test_score = evaluate_features(
            X_train_imp, X_test_imp, y_train, y_test, selected_features
        )
        
        results_importance.append({
            'threshold': threshold,
            'n_features': len(selected_features),
            'train_score': train_score,
            'test_score': test_score,
            'selected_features': selected_features
        })
        
        print(f"  Threshold={threshold}: {len(selected_features)} features, Test Accuracy = {test_score:.4f}")

# Mejor resultado por importancia
if results_importance:
    best_importance = max(results_importance, key=lambda x: x['test_score'])
    print(f"\n🏆 Mejor Feature Importance: {best_importance['n_features']} características con accuracy {best_importance['test_score']:.4f}")
    print(f"Características seleccionadas: {best_importance['selected_features'][:10]}...")

## 📊 Paso 5: Comparación de Métodos de Feature Selection

**🎯 Análisis**: Comparar los resultados de los diferentes métodos para entender sus fortalezas y debilidades.

In [ ]:
# Comparación de resultados
print("📊 COMPARACIÓN DE MÉTODOS DE FEATURE SELECTION\n")

# Preparar tabla de comparación
comparison_data = []

# Baseline
comparison_data.append({
    'Método': 'Baseline (Todas)',
    'N_Features': X_processed.shape[1],
    'Train_Accuracy': baseline_train,
    'Test_Accuracy': baseline_test,
    'Selected_Features': list(X_processed.columns)
})

# SelectKBest
if 'best_selectk' in locals():
    comparison_data.append({
        'Método': 'SelectKBest',
        'N_Features': best_selectk['k'],
        'Train_Accuracy': best_selectk['train_score'],
        'Test_Accuracy': best_selectk['test_score'],
        'Selected_Features': best_selectk['selected_features']
    })

# RFE
if 'best_rfe' in locals():
    comparison_data.append({
        'Método': 'RFE',
        'N_Features': best_rfe['n_features'],
        'Train_Accuracy': best_rfe['train_score'],
        'Test_Accuracy': best_rfe['test_score'],
        'Selected_Features': best_rfe['selected_features']
    })

# Feature Importance
if 'best_importance' in locals():
    comparison_data.append({
        'Método': 'Feature Importance',
        'N_Features': best_importance['n_features'],
        'Train_Accuracy': best_importance['train_score'],
        'Test_Accuracy': best_importance['test_score'],
        'Selected_Features': best_importance['selected_features']
    })

# Crear DataFrame para comparación
comparison_df = pd.DataFrame(comparison_data)
print(comparison_df[['Método', 'N_Features', 'Train_Accuracy', 'Test_Accuracy']].to_string(index=False))

# Visualización de comparación
plt.figure(figsize=(12, 8))
methods = comparison_df['Método']
test_accuracies = comparison_df['Test_Accuracy']
n_features = comparison_df['N_Features']

plt.subplot(2, 1, 1)
bars = plt.bar(methods, test_accuracies, color='skyblue', alpha=0.7)
plt.title('Comparación de Accuracy en Test por Método')
plt.ylabel('Test Accuracy')
plt.xticks(rotation=45)
for i, (bar, acc) in enumerate(zip(bars, test_accuracies)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, 
             f'{acc:.3f}', ha='center', va='bottom')

plt.subplot(2, 1, 2)
bars2 = plt.bar(methods, n_features, color='lightcoral', alpha=0.7)
plt.title('Número de Características Seleccionadas')
plt.ylabel('Número de Características')
plt.xticks(rotation=45)
for i, (bar, n_feat) in enumerate(zip(bars2, n_features)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1, 
             f'{n_feat}', ha='center', va='bottom')

plt.tight_layout()
plt.show()

## 🔍 Paso 6: Análisis de Overfitting con Alta Dimensionalidad

**🎯 Concepto Clave**: La maldición de la dimensionalidad. Observar cómo el rendimiento cambia con el número de características.

In [ ]:
# Análisis de Overfitting: Accuracy vs Número de Características
print("🔍 Análisis de Overfitting: Rendimiento vs Número de Características")

# Usar resultados de SelectKBest para el análisis
n_features_list = [item['k'] for item in results_selectk]
train_scores_list = [item['train_score'] for item in results_selectk]
test_scores_list = [item['test_score'] for item in results_selectk]

# Agregar baseline
n_features_list = [X_processed.shape[1]] + n_features_list
train_scores_list = [baseline_train] + train_scores_list
test_scores_list = [baseline_test] + test_scores_list

# Visualización
plt.figure(figsize=(12, 8))
plt.plot(n_features_list, train_scores_list, 'o-', label='Training Accuracy', linewidth=2, markersize=8)
plt.plot(n_features_list, test_scores_list, 's-', label='Test Accuracy', linewidth=2, markersize=8)
plt.xlabel('Número de Características')
plt.ylabel('Accuracy')
plt.title('Análisis de Overfitting: Accuracy vs Número de Características')
plt.legend()
plt.grid(True, alpha=0.3)

# Agregar anotaciones para puntos importantes
max_test_idx = test_scores_list.index(max(test_scores_list))
plt.annotate(f'Mejor Test: {max(test_scores_list[max_test_idx]):.3f}', 
             xy=(n_features_list[max_test_idx], test_scores_list[max_test_idx]),
             xytext=(10, 10), textcoords='offset points',
             bbox=dict(boxstyle='round,pad=0.3', facecolor='yellow', alpha=0.7),
             arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0'))

plt.tight_layout()
plt.show()

# Análisis de la brecha train-test (overfitting)
print("\n📊 Análisis de Overfitting:")
print(f"{'Método':<20} {'Train':<8} {'Test':<8} {'Gap':<8}")
print("-" * 45)

for i, method in enumerate(['Baseline'] + [f'SelectKBest-{k}' for k in n_features_list[1:]]):
    gap = train_scores_list[i] - test_scores_list[i]
    print(f"{method:<20} {train_scores_list[i]:<8.3f} {test_scores_list[i]:<8.3f} {gap:<8.3f}")

print("\n💡 Interpretación:")
print("- Gap grande = Overfitting (modelo memoriza datos de entrenamiento)")
print("- Gap pequeño = Modelo bien generalizado")
print("- El punto óptimo balancea rendimiento y generalización")

## 🎯 Paso 7: Reducción de Dimensionalidad con PCA

**🎯 Técnica Adicional**: PCA (Principal Component Analysis) como alternativa al feature selection.

**¿Cuándo usar PCA vs Feature Selection?**
- **Feature Selection**: Mantienes las características originales (interpretable)
- **PCA**: Creas nuevas características (no interpretable, pero puede ser más efectivo)

In [ ]:
# PCA: Reducción de dimensionalidad por componentes principales
print("🎯 TÉCNICA 4: Principal Component Analysis (PCA)")

# Probar diferentes números de componentes
n_components_list = [5, 10, 15, 20, 25, 30, 40, 50]
results_pca = []
explained_variance_ratio = []

for n_components in n_components_list:
    pca = PCA(n_components=n_components, random_state=42)
    X_train_pca = pca.fit_transform(X_train_scaled)
    X_test_pca = pca.transform(X_test_scaled)
    
    train_score, test_score = evaluate_features(
        X_train_pca, X_test_pca, y_train, y_test, [f'PC{i+1}' for i in range(n_components)]
    )
    
    results_pca.append({
        'n_components': n_components,
        'train_score': train_score,
        'test_score': test_score,
        'explained_variance': pca.explained_variance_ratio_.sum()
    })
    
    print(f"  Componentes: {n_components:2d} | Varianza explicada: {pca.explained_variance_ratio_.sum():.3f} | Test Acc: {test_score:.4f}")

# Visualizar varianza explicada acumulada
plt.figure(figsize=(15, 5))

# Subplot 1: Varianza explicada
plt.subplot(1, 3, 1)
n_comp_vals = [r['n_components'] for r in results_pca]
explained_vars = [r['explained_variance'] for r in results_pca]
plt.plot(n_comp_vals, explained_vars, 'o-', linewidth=2, markersize=6)
plt.xlabel('Número de Componentes')
plt.ylabel('Varianza Explicada Acumulada')
plt.title('Varianza Explicada por PCA')
plt.grid(True, alpha=0.3)

# Subplot 2: Accuracy
plt.subplot(1, 3, 2)
train_scores_pca = [r['train_score'] for r in results_pca]
test_scores_pca = [r['test_score'] for r in results_pca]
plt.plot(n_comp_vals, train_scores_pca, 'o-', label='Train', linewidth=2)
plt.plot(n_comp_vals, test_scores_pca, 's-', label='Test', linewidth=2)
plt.xlabel('Número de Componentes')
plt.ylabel('Accuracy')
plt.title('Accuracy con PCA')
plt.legend()
plt.grid(True, alpha=0.3)

# Subplot 3: Comparación con métodos anteriores
plt.subplot(1, 3, 3)
all_methods = ['Baseline', 'SelectKBest', 'Feature Importance']
all_accuracies = [baseline_test]

if 'best_selectk' in locals():
    all_accuracies.append(best_selectk['test_score'])
if 'best_importance' in locals():
    all_accuracies.append(best_importance['test_score'])

best_pca = max(results_pca, key=lambda x: x['test_score'])
all_accuracies.append(best_pca['test_score'])
all_methods.append('PCA')
all_n_features = [X_processed.shape[1]]

if 'best_selectk' in locals():
    all_n_features.append(best_selectk['k'])
if 'best_importance' in locals():
    all_n_features.append(best_importance['n_features'])
all_n_features.append(best_pca['n_components'])

bars = plt.bar(range(len(all_methods)), all_accuracies, alpha=0.7)
plt.xticks(range(len(all_methods)), all_methods, rotation=45)
plt.ylabel('Test Accuracy')
plt.title('Comparación Final de Métodos')
for i, (bar, acc) in enumerate(zip(bars, all_accuracies)):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001,
             f'{acc:.3f}\n({all_n_features[i]} feat)', ha='center', va='bottom', fontsize=8)

plt.tight_layout()
plt.show()

print(f"\n🏆 Mejor PCA: {best_pca['n_components']} componentes con accuracy {best_pca['test_score']:.4f}")
print(f"   Varianza explicada: {best_pca['explained_variance']:.3f}")

## 📋 Paso 8: Validación Cruzada del Mejor Modelo

**🎯 Validación Final**: Usar validación cruzada para obtener una estimación más robusta del rendimiento.

In [ ]:
# Validación cruzada del mejor modelo (seleccionado de los resultados anteriores)
print("📋 Validación Cruzada del Mejor Modelo")

# Determinar el mejor método basado en test accuracy
best_overall = max(comparison_data, key=lambda x: x['Test_Accuracy'])
best_pca_overall = max(results_pca, key=lambda x: x['test_score'])

if best_pca_overall['test_score'] > best_overall['Test_Accuracy']:
    print(f"🏆 Mejor método overall: PCA con {best_pca_overall['n_components']} componentes")
    final_method = 'PCA'
    final_n_features = best_pca_overall['n_components']
    final_score = best_pca_overall['test_score']
else:
    print(f"🏆 Mejor método overall: {best_overall['Método']} con {best_overall['N_Features']} características")
    final_method = best_overall['Método']
    final_n_features = best_overall['N_Features']
    final_score = best_overall['Test_Accuracy']

# Validación cruzada
from sklearn.model_selection import cross_val_score

print(f"\n🔄 Realizando validación cruzada (5-fold) para {final_method}...")

# Preparar datos según el mejor método
if final_method == 'PCA':
    # Re-ajustar PCA con todos los datos
    pca_final = PCA(n_components=final_n_features, random_state=42)
    X_final = pca_final.fit_transform(scaler.fit_transform(X_processed))
    y_final = y_encoded
elif final_method == 'SelectKBest':
    selector_final = SelectKBest(score_func=f_classif, k=final_n_features)
    X_final = selector_final.fit_transform(scaler.fit_transform(X_processed), y_encoded)
    y_final = y_encoded
elif final_method == 'Feature Importance':
    threshold = best_importance['threshold']
    selected_features = feature_importance[feature_importance['importance'] >= threshold]['feature'].tolist()
    feature_indices = [X_processed.columns.get_loc(feature) for feature in selected_features]
    X_final = scaler.fit_transform(X_processed)[:, feature_indices]
    y_final = y_encoded
else:  # Baseline
    X_final = scaler.fit_transform(X_processed)
    y_final = y_encoded

# Validación cruzada
rf_cv = RandomForestClassifier(n_estimators=100, random_state=42)
cv_scores = cross_val_score(rf_cv, X_final, y_final, cv=5, scoring='accuracy')

print(f"\n📊 Resultados de Validación Cruzada (5-fold):")
print(f"  - Accuracy promedio: {cv_scores.mean():.4f} ± {cv_scores.std():.4f}")
print(f"  - Scores individuales: {cv_scores}")
print(f"  - Rango: [{cv_scores.min():.4f}, {cv_scores.max():.4f}]")

# Comparación con split train-test original
print(f"\n📈 Comparación:")
print(f"  - Hold-out Test Split: {final_score:.4f}")
print(f"  - CV Promedio: {cv_scores.mean():.4f}")
print(f"  - Diferencia: {abs(final_score - cv_scores.mean()):.4f}")

if abs(final_score - cv_scores.mean()) < 0.01:
    print("  ✅ Consistencia alta entre métodos de validación")
else:
    print("  ⚠️ Diferencia notable - considerar más folds en CV")

## 🎓 Paso 9: Conclusiones y Recomendaciones

**🎯 Síntesis de Aprendizajes**: Resumen de las técnicas aprendidas y cuándo aplicarlas.

In [ ]:
# Resumen final y conclusiones
print("🎓 CONCLUSIONES DEL ANÁLISIS DE FEATURE SELECTION")
print("=" * 55)

print("\n📊 RESUMEN DE MÉTODOS PROBADOS:")
for i, item in enumerate(comparison_data, 1):
    method = item['Método']
    n_feat = item['N_Features']
    test_acc = item['Test_Accuracy']
    gap = item['Train_Accuracy'] - item['Test_Accuracy']
    
    print(f"  {i}. {method}:")
     - Características: {n_feat}")
     - Test Accuracy: {test_acc:.4f}")
     - Gap (overfitting): {gap:.4f}")

print(f"\n🏆 MEJOR MÉTODO: {final_method}")
print(f"   - Características/Componentes: {final_n_features}")
print(f"   - Reducción de dimensionalidad: {((X_processed.shape[1] - final_n_features) / X_processed.shape[1] * 100):.1f}%")
print(f"   - Accuracy final: {final_score:.4f}")

print("\n💡 RECOMENDACIONES PARA CASOS DE USO:")
print("\n🔍 Cuándo usar cada método:")
print("  • SelectKBest: Datos con características independientes, necesitas interpretar resultados")
print("  • RFE: Cuando tienes un estimador específico en mente, quieres el mejor subconjunto")
print("  • Feature Importance: Usando Random Forest/XGBoost, balance entre rendimiento e interpretabilidad")
print("  • PCA: Cuando las características están correlacionadas, no necesitas interpretabilidad")

print("\n⚠️ SEÑALES DE WARNING:")
print("  • Gap train-test > 0.1 → Posible overfitting")
print("  • Accuracy decrece abruptamente → Umbral de características alcanzado")
print("  • Varianza explicada < 80% con PCA → Considera más componentes")

print("\n🚀 SIGUIENTES PASOS SUGERIDOS:")
print("  1. Implementa GridSearch para optimizar hiperparámetros del mejor modelo")
print("  2. Prueba otros algoritmos (XGBoost, SVM) con las características seleccionadas")
print("  3. Analiza la importancia de las características en el contexto del dominio")
print("  4. Considera ensemble de diferentes métodos de feature selection")

print(f"\n✅ DATASET COMPLETADO: Forest Cover Type")
print(f"   - Técnicas aprendidas: Feature Selection, Alta Dimensionalidad, PCA")
print(f"   - Próximo dataset sugerido: Adult Census (manejo de strings + valores faltantes)")

---

## 📚 Resumen de Técnicas Aplicadas

| Técnica | Propósito | Cuándo Usar | Resultado en Este Dataset |
|---------|-----------|-------------|---------------------------|
| **Correlación** | Identificar características redundantes | Antes de feature selection | 10+ pares correlacionados |
| **SelectKBest** | Selección univariada por score | Características independientes | Accuracy similar con 30% de features |
| **RFE** | Selección recursiva | Estimador específico disponible | Mejor rendimiento con 20 features |
| **Feature Importance** | Basado en Random Forest | Usando ensemble methods | Buena interpretabilidad, rendimiento óptimo |
| **PCA** | Reducción por componentes | Datos correlacionados, sin interpretabilidad | Similar rendimiento, menos interpretabilidad |

## 🎯 Progresión de Dificultad del Curso

1. **Wine Quality** (Semana 1) - ⭐ Básicos: 13 features, limpieza simple
2. **Forest Cover Type** (Semana 2) - ⭐⭐⭐ **Avanzado**: 54 features, feature selection
3. **Adult Census** (Semana 3) - ⭐⭐ Strings + Missing values
4. **Heart Disease** (Semana 4) - ⭐⭐⭐ Imputación avanzada

**Próximo notebook**: Adult Census Income Dataset para aprender manejo de strings y valores faltantes.